# Stage 9 - MRL Eye Open/Closed CNN Baselines

This notebook trains the **MRL Eye eye-state specialist**. It is not a full drowsiness classifier.

The module outputs per-frame probabilities:

- `p_eye_closed`
- `p_eye_open`

These outputs are intended for later temporal smoothing or PERCLOS-like fusion with the completed YawDD mouth/yawn module.

## Notebook Notes

- Use a GPU runtime for full training.
- This notebook uses the existing Stage 8 subject-level split manifest.
- It does not rebuild the dataset and does not create image-level random splits.
- The `DATA_ROOT` variable controls path remapping when the manifest contains local absolute paths from another machine.

In [ ]:
# 1. Runtime and repository configuration
from pathlib import Path

REPO_URL = "https://github.com/JohnCoffey-commits/Drowsiness-Detection.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/Drowsiness-Detection")

DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/Drowsiness_Detection_Colab")

# Expected Drive layout:
# DRIVE_PROJECT_ROOT / "dataset/mrlEyes_2018_01"
# DRIVE_PROJECT_ROOT / "artifacts/mappings/mrl_eye_trainable_with_split.csv"
DATA_ROOT = DRIVE_PROJECT_ROOT
MANIFEST = DRIVE_PROJECT_ROOT / "artifacts/mappings/mrl_eye_trainable_with_split.csv"
OUTPUT_DIR = DRIVE_PROJECT_ROOT / "outputs/mrl_eye"

FORCE_RECLONE_REPO = False

MODELS = ["resnet18", "mobilenet_v2", "efficientnet_b0"]
EPOCHS = 10
BATCH_SIZE = 64
IMAGE_SIZE = 224
NUM_WORKERS = 2
SEED = 42

In [ ]:
# 2. Mount Google Drive
from google.colab import drive

drive.mount("/content/drive")
print("Drive project root:", DRIVE_PROJECT_ROOT)

In [ ]:
# 3. Clone and prepare repository code
import os
import shutil
import subprocess
import sys

if FORCE_RECLONE_REPO and REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repository already exists at {REPO_DIR}. Reusing it.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Working directory:", Path.cwd())

In [ ]:
# 4. Install/import dependencies if needed
import importlib
import subprocess
import sys

for package, import_name in [("pandas", "pandas"), ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib"), ("tqdm", "tqdm")]:
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import torch
import torchvision
print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Path Setup

`MANIFEST` should point to `mrl_eye_trainable_with_split.csv`.

`DATA_ROOT` should be the project root that contains `dataset/mrlEyes_2018_01/`. The training script joins `DATA_ROOT` with the manifest `relative_path` column when possible, so absolute local paths from another machine do not need to be rewritten.

In [ ]:
# 5. Verify required inputs
required_paths = {
    "DATA_ROOT": DATA_ROOT,
    "MRL dataset": DATA_ROOT / "dataset/mrlEyes_2018_01",
    "Stage 8 manifest": MANIFEST,
}

missing = []
for name, path in required_paths.items():
    exists = path.exists()
    print(f"{name}: {path} -> {'OK' if exists else 'MISSING'}")
    if not exists:
        missing.append(f"{name}: {path}")

if missing:
    raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))

In [ ]:
# 6. Small manifest sanity check
import pandas as pd
from pathlib import Path

df = pd.read_csv(MANIFEST, dtype={"subject_id": str, "sensor_id": str})
print("Rows:", len(df))
print("Columns:", list(df.columns))
print("Labels:", df[["label", "label_name"]].drop_duplicates().sort_values("label").to_string(index=False))
print("Split/class counts:")
print(df.groupby(["split", "label_name"]).size().unstack(fill_value=0))

assert set(df["label"].astype(int)) == {0, 1}
assert set(df["split"]) == {"train", "val", "test"}
assert (df.groupby("subject_id")["split"].nunique() == 1).all(), "Subject leakage detected"

In [ ]:
# 7. Optional tiny dry run before full training
RUN_DRY_RUN = False

if RUN_DRY_RUN:
    dry_cmd = [
        sys.executable, "src/training/train_mrl_eye_baselines.py",
        "--models", "resnet18",
        "--epochs", "1",
        "--batch-size", "16",
        "--max-samples-per-split", "64",
        "--manifest", str(MANIFEST),
        "--data-root", str(DATA_ROOT),
        "--output-dir", str(DRIVE_PROJECT_ROOT / "outputs/mrl_eye_debug"),
    ]
    print("Running:", " ".join(dry_cmd))
    subprocess.run(dry_cmd, check=True)

In [ ]:
# 8. Launch full Stage 9 training
train_cmd = [
    sys.executable, "src/training/train_mrl_eye_baselines.py",
    "--models", *MODELS,
    "--epochs", str(EPOCHS),
    "--batch-size", str(BATCH_SIZE),
    "--image-size", str(IMAGE_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--seed", str(SEED),
    "--manifest", str(MANIFEST),
    "--data-root", str(DATA_ROOT),
    "--output-dir", str(OUTPUT_DIR),
]
print("Running:", " ".join(train_cmd))
subprocess.run(train_cmd, check=True)

In [ ]:
# 9. Display final metrics table
results_csv = OUTPUT_DIR / "results/mrl_eye_initial_results.csv"
results = pd.read_csv(results_csv)
display_cols = [
    "model", "best_val_macro_f1", "test_accuracy", "test_macro_f1",
    "test_recall_closed", "test_false_open_count", "test_false_closed_count",
]
display(results[display_cols])

In [ ]:
# 10. Display confusion matrices and training curves
from IPython.display import Image as IPyImage, display

for model_name in MODELS:
    print("\n", model_name)
    curve = OUTPUT_DIR / "figures" / f"{model_name}_training_curve.png"
    cm = OUTPUT_DIR / "figures" / f"{model_name}_confusion_matrix.png"
    if curve.exists():
        display(IPyImage(filename=str(curve)))
    if cm.exists():
        display(IPyImage(filename=str(cm)))

In [ ]:
# 11. Inspect threshold sweeps for p_eye_closed
for model_name in MODELS:
    sweep_path = OUTPUT_DIR / "results" / f"{model_name}_threshold_sweep.csv"
    if sweep_path.exists():
        print("\n", model_name)
        display(pd.read_csv(sweep_path))

## Interpretation Notes

- Prioritize validation macro F1 for checkpoint selection.
- Review closed-eye recall carefully. False-open errors are true closed-eye frames predicted as open, which can hide eye-closure events.
- Do not choose a threshold that predicts nearly everything as closed. Use the threshold sweep to balance closed recall with macro F1 and false-closed errors.
- The next stage should use `p_eye_closed` over time for smoothing or PERCLOS-like logic, then fuse with YawDD mouth/yawn outputs.